# Python Exception Handling Exercises: 20 Coding Problems with Solutions

A practice notebook on try/except/else/finally, custom exceptions, chaining, context managers, retry decorators, threads, and generators — each with a concept note, a hint, a solution, and an explanation.

*Adapted for practice from the exercise list at [PYnative](https://pynative.com/python-exception-handling-exercises/). A couple of exercises that use interactive `input()` or reference files from earlier exercises have been adapted to run standalone and non-interactively in this notebook.*

---

## Concepts you'll need

This set goes from basic `try`/`except` through custom exceptions, chaining, decorators, threads, and generators.

- **The full clause order** — `try` → `except` (one or more, most specific first) → `else` (runs only if no exception occurred) → `finally` (always runs, for cleanup). Catch specific exception types rather than a bare `except` so unrelated bugs aren't silently swallowed.
- **Built-in exceptions used here** — `ValueError` (bad conversion, e.g. `int("abc")`), `ZeroDivisionError`, `FileNotFoundError`/`IOError`, `IndexError`, `KeyError`, `TypeError`.
- **Re-raising** — a bare `raise` (no argument) inside an `except` block re-raises the exact same exception with its original traceback intact — different from `raise e`, which starts a new traceback.
- **Custom exceptions** — `class MyError(Exception): ...` lets callers catch exactly your domain's errors instead of a generic built-in type; a hierarchy (`class Specific(General): ...`) lets a handler catch broadly or narrowly as needed.
- **Exception chaining** — `raise NewError(...) from original_exception` sets `new_error.__cause__`, preserving the technical root cause while presenting a higher-level, domain-specific error to the caller.
- **Context managers** — `__enter__`/`__exit__` power the `with` statement; returning `True` from `__exit__` suppresses the exception it was given, `False`/`None` lets it propagate.
- **`logging.exception()`** — call from inside an `except` block to log an error *with its full traceback* automatically, more suited to production code than bare `print()`.
- **Decorators with arguments** — a retry decorator needs three nested function levels: `retry(times)` (the factory) → `decorator(func)` → `wrapper(*args, **kwargs)` (what actually runs); `functools.wraps(func)` preserves the original function's name/docstring on the wrapper.
- **Threads and generators** — exceptions raised inside a `threading.Thread` never automatically reach the main thread; they must be caught and stored (behind a `Lock` if shared) for the main thread to see. In a generator, `gen.throw(ExcType, msg)` resumes it at its current `yield` and raises the exception there — the standard way to inject a cancellation signal into a paused generator.

Each exercise below gives a problem, a hint, a solution, and an explanation.

## Exercise 1. Basic Try-Except

**Concept:** the fundamental try/except pattern

**Problem:** Handle invalid (non-integer) user input without crashing.

**Given:**
```
user enters "hello" when asked for a number
```

**Expected Output:**
```
Error: That was not a valid integer. Please enter a number.
```

**Hint:** int() raises ValueError automatically on a non-numeric string — no need to raise it yourself.

In [ ]:
try:
    user_input = input("Enter a number: ")
    number = int(user_input)
    print("You entered:", number)
except ValueError:
    print("Error: That was not a valid integer. Please enter a number.")

**Explanation:** Python attempts every line inside try; the moment int(user_input) raises ValueError, execution jumps straight to the matching except block. Naming the specific exception type (rather than a bare except:) means only genuine conversion failures are caught — any unrelated error would still surface normally.

## Exercise 2. Division Safety

**Concept:** returning a sentinel value (None) instead of crashing

**Problem:** Write a safe_divide() that returns None instead of raising on division by zero.

**Given:**
```
safe_divide(10, 0) and safe_divide(10, 2)
```

**Expected Output:**
```
5.0
None
```

**Hint:** Catch only ZeroDivisionError specifically, so other errors (like TypeError) still propagate normally.

In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
        return result
    except ZeroDivisionError:
        print("Warning: Division by zero is not allowed.")
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

**Explanation:** The division only raises when b is 0, at which point result is never assigned and control jumps to except. Returning None as a sentinel lets the caller check `if result is None:` afterward without needing its own try/except — a common pattern for functions where failure is an expected, recoverable outcome.

## Exercise 3. File Not Found

**Concept:** FileNotFoundError with a with statement

**Problem:** Attempt to open a missing file and show a friendly message instead of a raw traceback.

**Given:**
```
filename = "missing_file.txt"
```

**Expected Output:**
```
Error: The file 'missing_file.txt' was not found. Please check the filename and path.
```

**Hint:** with open(...) always closes the file handle even if an exception is raised inside the block.

In [ ]:
filename = "missing_file.txt"

try:
    with open(filename, "r") as f:
        content = f.read()
        print(content)
except FileNotFoundError:
    print(f"Error: The file '{filename}' was not found. Please check the filename and path.")

**Explanation:** FileNotFoundError is a specific subclass of OSError raised only when the target path doesn't exist — catching this exact type avoids accidentally masking a different I/O problem like a permissions error. Embedding {filename} in the message gives the user something actionable rather than a bare traceback.

## Exercise 4. Index Out of Range

**Concept:** IndexError, plus stacking multiple except clauses

**Problem:** Safely access a list by a user-provided index, handling both bad input and out-of-range values.

**Given:**
```
items = ["apple", "banana", "cherry"]; user enters 5
```

**Expected Output:**
```
Error: Index 5 is out of range. The list has 3 items (valid indices: 0 to 2).
```

**Hint:** Stack two except clauses after one try — Python checks them top to bottom and runs the first match.

In [ ]:
items = ["apple", "banana", "cherry"]

try:
    index = int(input("Enter an index: "))
    print("Item:", items[index])
except ValueError:
    print("Error: Please enter a valid integer index.")
except IndexError:
    print(f"Error: Index {index} is out of range. "
          f"The list has {len(items)} items (valid indices: 0 to {len(items) - 1}).")

**Explanation:** int(input(...)) can fail with ValueError before the list is even touched; items[index] separately can fail with IndexError if the (valid) integer is out of bounds. Stacking both except clauses after one try lets each failure mode get its own tailored message, and len(items) - 1 keeps the reported valid range accurate even if the list's size changes later.

## Exercise 5. Key Error Handling

**Concept:** KeyError vs. dict.get() as an alternative

**Problem:** Look up a country's capital, handling the case where the country isn't in the dictionary.

**Given:**
```
capitals = {"France": "Paris", "Japan": "Tokyo", "Brazil": "Brasilia"}; user enters "Germany"
```

**Expected Output:**
```
Error: 'Germany' was not found in the dictionary.
```

**Hint:** Bracket access capitals[country] raises KeyError on a missing key; .get(country) would instead return None silently.

In [ ]:
capitals = {
    "France": "Paris",
    "Japan": "Tokyo",
    "Brazil": "Brasilia"
}

try:
    country = input("Enter a country name: ")
    capital = capitals[country]
    print(f"The capital of {country} is {capital}.")
except KeyError:
    print(f"Error: '{country}' was not found in the dictionary.")

**Explanation:** Bracket access is the idiomatic choice when a missing key genuinely represents an error condition worth reporting, whereas .get(key) suits cases where absence is normal and expected. Note dictionary keys are case-sensitive, so 'germany' would also trigger the KeyError — normalizing input with .title() could improve robustness if that mattered.

## Exercise 6. Type Error Guard

**Concept:** TypeError from incompatible operand types

**Problem:** Add two values safely, handling the case where their types don't support addition together.

**Given:**
```
add_values(10, "20") and add_values(10, 20)
```

**Expected Output:**
```
Error: incompatible types for addition.
30
```

**Hint:** + raises TypeError automatically when applied to genuinely incompatible types like int and str.

In [ ]:
def add_values(a, b):
    try:
        return a + b
    except TypeError:
        return "Error: incompatible types for addition."

print(add_values(10, 20))
print(add_values(10, "20"))

**Explanation:** Python won't silently coerce a string into a number for +, so mixing int and str raises TypeError immediately. Catching only TypeError (not a bare except) means an unrelated error type, like MemoryError, would still propagate rather than being silently swallowed by this function.

## Exercise 7. Multiple Exceptions

**Concept:** two distinct except clauses, each with its own message

**Problem:** Convert a string to a float and divide it, giving separate messages for conversion failure vs. division by zero.

**Given:**
```
parse_and_divide("abc", 2), parse_and_divide("10", 0), parse_and_divide("10", 2)
```

**Expected Output:**
```
Error: 'abc' cannot be converted to a number.
Error: Cannot divide by zero.
5.0
```

**Hint:** Put both risky lines in the same try block; Python stops at whichever line fails first.

In [ ]:
def parse_and_divide(value, divisor):
    try:
        number = float(value)
        result = number / divisor
        return result
    except ValueError:
        return f"Error: '{value}' cannot be converted to a number."
    except ZeroDivisionError:
        return "Error: Cannot divide by zero."

print(parse_and_divide("abc", 2))
print(parse_and_divide("10", 0))
print(parse_and_divide("10", 2))

**Explanation:** float(value) fails first with ValueError for non-numeric input, before the division line is ever reached; if conversion succeeds but divisor is 0, ZeroDivisionError fires on the next line instead. Since the two failure messages differ, they're kept as separate except clauses rather than combined into one `except (ValueError, ZeroDivisionError):` tuple form.

## Exercise 8. Finally Block

**Concept:** guaranteed cleanup with finally, without using `with`

**Problem:** Manually open and close a file, guaranteeing the close happens whether or not an error occurs.

**Given:**
```
read_file("hello.txt") (exists) and read_file("missing.txt") (doesn't)
```

**Expected Output:**
```
File closed.
Hello from the file!
---
File closed.
Error: 'missing.txt' not found.
```

**Hint:** Initialize f = None before try, so finally can safely check `if f:` before calling f.close().

In [ ]:
with open("hello.txt", "w") as setup_file:
    setup_file.write("Hello from the file!")

def read_file(filename):
    f = None
    try:
        f = open(filename, "r")
        content = f.read()
        return content
    except FileNotFoundError:
        return f"Error: '{filename}' not found."
    finally:
        if f:
            f.close()
        print("File closed.")

print(read_file("hello.txt"))
print("---")
print(read_file("missing.txt"))

**Explanation:** finally runs unconditionally after try and any matching except, regardless of success or failure — making it the correct home for cleanup logic like closing a file handle. Initializing f = None beforehand matters: if open() itself is what raised the exception, f would never have been assigned, so `if f:` avoids a secondary AttributeError when finally tries to close it. (In practice, `with open(...) as f:` handles this automatically — this exercise shows what it's doing under the hood.)

## Exercise 9. Else Clause

**Concept:** try/except/else — keeping the happy path separate

**Problem:** Compute a square root, using else so success logic isn't accidentally shielded by the except clause.

**Given:**
```
safe_sqrt("25") and safe_sqrt("abc")
```

**Expected Output:**
```
Success: the square root of 25.0 is 5.0
Error: 'abc' is not a valid number.
```

**Hint:** Only the risky conversion goes in try; the else block runs only when that conversion succeeded.

In [ ]:
import math

def safe_sqrt(value):
    try:
        number = float(value)
    except ValueError:
        print(f"Error: '{value}' is not a valid number.")
    else:
        result = math.sqrt(number)
        print(f"Success: the square root of {number} is {result}")

safe_sqrt("25")
safe_sqrt("abc")

**Explanation:** Only float(value) sits inside try — if math.sqrt() were also inside try, a negative-number error from sqrt() would incorrectly get caught by the except ValueError clause meant for conversion failures. else runs only when try completed with no exception at all, keeping the 'happy path' logic cleanly separated from the specific error it's guarding against.

## Exercise 10. Nested Try-Except

**Concept:** an outer try/except wrapping an inner one, for independent failure points

**Problem:** Open a JSON file (outer risk) and look up a key in it (inner, separate risk), each with its own handler.

**Given:**
```
data.json = {"name": "Alice", "age": 30}
```

**Expected Output:**
```
Alice
Error: key 'email' not found in the data.
Error: file 'missing.json' does not exist.
```

**Hint:** The inner try only runs once the outer try (opening + parsing the file) has already succeeded.

In [ ]:
import json

with open("data.json", "w") as setup_file:
    json.dump({"name": "Alice", "age": 30}, setup_file)

def load_and_parse(filename, key):
    try:
        with open(filename, "r") as f:
            data = json.load(f)
        try:
            value = data[key]
            print(value)
        except KeyError:
            print(f"Error: key '{key}' not found in the data.")
    except FileNotFoundError:
        print(f"Error: file '{filename}' does not exist.")

load_and_parse("data.json", "name")
load_and_parse("data.json", "email")
load_and_parse("missing.json", "name")

**Explanation:** If the file simply doesn't exist, the outer except catches it immediately and the inner block never even runs — there's no point checking for a key inside data that was never loaded. The inner try isolates the separate, independent risk of a missing dictionary key, so each failure point gets its own precisely-targeted handler rather than one handler trying to cover both cases.

## Exercise 11. Re-raising Exceptions

**Concept:** a bare `raise` to log-then-delegate to the caller

**Problem:** Log an exception at the point it occurs, then let the caller handle it too via a bare raise.

**Given:**
```
process_data("abc") called from within a wrapping try block
```

**Expected Output:**
```
[log] process_data failed: invalid literal for int() with base 10: 'abc'
[main] Caught re-raised exception: invalid literal for int() with base 10: 'abc'
```

**Hint:** A bare `raise` (no argument) re-raises the exact same exception with its original traceback intact — unlike `raise e`.

In [ ]:
def process_data(value):
    try:
        return int(value)
    except ValueError as e:
        print(f"[log] process_data failed: {e}")
        raise

try:
    process_data("abc")
except ValueError as e:
    print(f"[main] Caught re-raised exception: {e}")

**Explanation:** `except ValueError as e:` binds the exception object to e, so its message can be logged via str(e) or an f-string. A bare raise (no argument) re-raises the currently-active exception unchanged, preserving its original traceback — this differs from `raise e`, which would start a fresh traceback at that line, losing information about where the error actually originated.

## Exercise 12. Custom Exception Class

**Concept:** defining a domain-specific exception via `class X(Exception)`

**Problem:** Create a custom exception carrying structured context, raised when a withdrawal exceeds the balance.

**Given:**
```
BankAccount(100), attempt to withdraw 150
```

**Expected Output:**
```
Error: Cannot withdraw 150. Available balance: 100.
```

**Hint:** super().__init__(message) passes a human-readable string up to the base Exception class.

In [ ]:
class InsufficientFundsError(Exception):
    def __init__(self, amount, balance):
        self.amount = amount
        self.balance = balance
        super().__init__(
            f"Cannot withdraw {amount}. Available balance: {balance}."
        )

class BankAccount:
    def __init__(self, balance):
        self.balance = balance

    def withdraw(self, amount):
        if amount > self.balance:
            raise InsufficientFundsError(amount, self.balance)
        self.balance -= amount
        print(f"Withdrew {amount}. New balance: {self.balance}.")

account = BankAccount(100)

try:
    account.withdraw(150)
except InsufficientFundsError as e:
    print(f"Error: {e}")

**Explanation:** Inheriting from Exception is the standard way to define a custom error type. Storing amount and balance as attributes (not just baking them into the message string) means a caller could inspect e.amount or e.balance programmatically for fine-grained recovery. A custom exception type also lets callers catch exactly this domain error without risking catching an unrelated ValueError raised somewhere else in the system.

## Exercise 13. Exception Chaining

**Concept:** raise ... from ... to preserve the original cause

**Problem:** Translate a low-level FileNotFoundError into a higher-level domain error, keeping the original cause accessible.

**Given:**
```
load_config("config.json") when the file is missing
```

**Expected Output:**
```
ConfigurationError: Could not load config file 'config.json'.
Caused by: [Errno 2] No such file or directory: 'config.json'
```

**Hint:** `from e` attaches the original exception as the new one's __cause__, visible via e.__cause__.

In [ ]:
import json

class ConfigurationError(Exception):
    pass

def load_config(filename):
    try:
        with open(filename, "r") as f:
            return json.load(f)
    except FileNotFoundError as e:
        raise ConfigurationError(
            f"Could not load config file '{filename}'."
        ) from e

try:
    load_config("config.json")
except ConfigurationError as e:
    print(f"ConfigurationError: {e}")
    print(f"Caused by: {e.__cause__}")

**Explanation:** `raise NewException(...) from e` explicitly sets new_exception.__cause__ to e, and marks the chain as intentional (as opposed to incidental __context__ chaining, which happens automatically if you raise inside an except without `from`). This lets a module present a clean, business-level error (ConfigurationError) at its public boundary while still preserving the low-level technical root cause (FileNotFoundError) for anyone who needs to debug further via e.__cause__.

## Exercise 14. Validate User Input Loop

**Concept:** combining try/except with while True, continue, and break

**Problem:** Keep prompting until the user enters a valid positive integer.

**Given:**
```
user types 'abc', then '-5', then '7'
```

**Expected Output:**
```
Error: 'abc' is not a valid integer. Try again.
Error: -5 is not positive. Try again.
You entered: 7
```

**Hint:** continue restarts the loop on failure; break only fires once both the type check and the value check pass.

In [ ]:
import builtins
_inputs = iter(["abc", "-5", "7"])
_orig_input = builtins.input
def _fake_input(prompt=""):
    val = next(_inputs)
    print(f"{prompt}{val}")
    return val
builtins.input = _fake_input

while True:
    try:
        raw = input("Enter a positive integer: ")
        number = int(raw)
    except ValueError:
        print(f"Error: '{raw}' is not a valid integer. Try again.")
        continue
    if number <= 0:
        print(f"Error: {number} is not positive. Try again.")
        continue
    print(f"You entered: {number}")
    break

builtins.input = _orig_input

**Explanation:** while True creates the 'keep trying until success' loop, exited only by the break at the very end. When int(raw) fails, continue jumps straight back to the top of the loop, skipping the positivity check entirely; when the conversion succeeds but the value is non-positive, a second continue does the same, this time via a plain if rather than an exception, since that's an expected business-rule failure rather than a programming error. (This notebook simulates the three user inputs automatically so the cell runs without prompting.)

## Exercise 15. Context Manager With Exceptions

**Concept:** a custom __enter__/__exit__ that selectively suppresses exceptions

**Problem:** Build a context manager that silently suppresses one exception type while letting others propagate normally.

**Given:**
```
SuppressError(ValueError) wrapping a block that raises ValueError, then a second block raising TypeError
```

**Expected Output:**
```
Entering context...
ValueError suppressed.
Outside context - execution continued.
Entering context...
(TypeError propagates normally)
```

**Hint:** Returning True from __exit__ suppresses the exception; False or None lets it propagate.

In [ ]:
class SuppressError:
    def __init__(self, exception_type):
        self.exception_type = exception_type

    def __enter__(self):
        print("Entering context...")
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        if exc_type is not None and issubclass(exc_type, self.exception_type):
            print(f"{exc_type.__name__} suppressed.")
            return True
        return False

with SuppressError(ValueError):
    raise ValueError("bad value")
print("Outside context - execution continued.")

try:
    with SuppressError(ValueError):
        result = 1 + "two"
except TypeError as e:
    print(f"TypeError propagated as expected: {e}")

**Explanation:** __exit__ receives the exception's type, value, and traceback whenever the with block raises one — all three are None if it completed normally. issubclass(exc_type, self.exception_type) (rather than an exact == comparison) mirrors how a real except clause behaves, also catching subclasses of the target type. The single most important rule: returning True tells Python the exception has been handled and should be suppressed, while False or None lets it propagate exactly as if the with block weren't there — which is why the TypeError from '1 + "two"' still escapes and needed its own try/except here to demonstrate that.

## Exercise 16. Exception Hierarchy

**Concept:** a 3-level custom exception chain, caught at different levels

**Problem:** Build AppError -> DatabaseError -> AppConnectionError, and show that catching any ancestor also catches the most specific one.

**Given:**
```
raise AppConnectionError("host unreachable"), caught three different ways
```

**Expected Output:**
```
Caught as ConnectionError: host unreachable
Caught as DatabaseError: host unreachable
Caught as AppError: host unreachable
```

**Hint:** except uses isinstance semantics — catching any ancestor in the chain intercepts a more specific exception.

In [ ]:
class AppError(Exception):
    pass

class DatabaseError(AppError):
    pass

class AppConnectionError(DatabaseError):
    pass

try:
    raise AppConnectionError("host unreachable")
except AppConnectionError as e:
    print(f"Caught as ConnectionError: {e}")

try:
    raise AppConnectionError("host unreachable")
except DatabaseError as e:
    print(f"Caught as DatabaseError: {e}")

try:
    raise AppConnectionError("host unreachable")
except AppError as e:
    print(f"Caught as AppError: {e}")

**Explanation:** AppConnectionError is-a DatabaseError is-a AppError is-a Exception, and Python's except clause checks using isinstance-style logic — so catching any ancestor along that chain successfully intercepts the more specific exception. A body of just `pass` is perfectly valid for simple marker exceptions, since the inherited Exception.__init__ already handles storing and displaying the message. (Note: the class was named AppConnectionError rather than ConnectionError specifically to avoid shadowing Python's own built-in ConnectionError.)

## Exercise 17. Logging Exceptions

**Concept:** logging.exception() for automatic traceback capture

**Problem:** Log a caught exception's full traceback to a file, rather than just printing a message.

**Given:**
```
divide(10, 0)
```

**Expected Output:**
```
Console: divide(10, 0) returned None
app.log: an ERROR entry with the full traceback
```

**Hint:** logging.exception() must be called from inside an active except block; it automatically attaches the current traceback.

In [ ]:
import logging

logging.basicConfig(
    filename="app.log",
    level=logging.ERROR,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True,
)

def divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        logging.exception("Division failed: attempted to divide %s by zero.", a)
        return None

result = divide(10, 0)
print(f"divide(10, 0) returned {result}")

result = divide(10, 2)
print(f"divide(10, 2) returned {result}")

with open("app.log") as f:
    print("\n--- app.log contents ---")
    print(f.read())

**Explanation:** logging.basicConfig() sets up a file handler, minimum severity level, and a format string once at the top of the script; %(asctime)s adds a timestamp to every entry. logging.exception(msg) logs at ERROR level and automatically appends the currently-active exception's full traceback — it's simply a more concise way of writing logging.error(msg, exc_info=True), and must be called from within the except block so Python knows which exception to attach.

## Exercise 18. Retry Decorator

**Concept:** a three-level decorator factory for automatic retries

**Problem:** Build a @retry(times=3) decorator that re-attempts a failing function and re-raises only after all attempts are exhausted.

**Given:**
```
a function that fails on its first two calls, succeeds on the third
```

**Expected Output:**
```
Attempt 1 failed: temporary failure
Attempt 2 failed: temporary failure
Attempt 3 succeeded.
Result: success
```

**Hint:** Three nesting levels: retry(times) is the factory, decorator(func) wraps the target, wrapper(*args, **kwargs) runs at call time.

In [ ]:
import functools

def retry(times=3):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(1, times + 1):
                try:
                    result = func(*args, **kwargs)
                    return result
                except Exception as e:
                    last_exception = e
                    print(f"Attempt {attempt} failed: {e}")
            raise last_exception
        return wrapper
    return decorator

call_count = 0

@retry(times=3)
def flaky_call():
    global call_count
    call_count += 1
    if call_count < 3:
        raise RuntimeError("temporary failure")
    print(f"Attempt {call_count} succeeded.")
    return "success"

print("Result:", flaky_call())

**Explanation:** retry(times) is the factory that captures the times argument via closure; decorator(func) is the actual decorator applied to flaky_call; wrapper(*args, **kwargs) is what actually executes on every call, looping up to times attempts. @functools.wraps(func) copies the original function's __name__ and __doc__ onto wrapper, so debugging tools show 'flaky_call' rather than the generic 'wrapper'. Storing the most recent failure in last_exception is essential — without it, the exception would go out of scope once the loop ends and there'd be nothing left to re-raise after all attempts fail.

## Exercise 19. Thread-Safe Exception Handling

**Concept:** catching exceptions inside threads and reporting them from the main thread

**Problem:** Run five threads that each divide, some by zero, and collect all failures safely into a shared list.

**Given:**
```
threads dividing 10 by [2, 0, 5, 0, 1]
```

**Expected Output:**
```
Successful divisions print immediately; two ZeroDivisionErrors get collected and reported afterward
```

**Hint:** Exceptions inside a Thread's target function never reach the main thread automatically — they must be caught and stored explicitly, behind a Lock if shared.

In [ ]:
import threading

errors = []
lock = threading.Lock()

def divide(thread_name, a, b):
    try:
        result = a / b
        print(f"{thread_name}: {a} / {b} = {result}")
    except ZeroDivisionError as e:
        with lock:
            errors.append((thread_name, e))

divisors = [2, 0, 5, 0, 1]
threads = []

for i, divisor in enumerate(divisors, start=1):
    t = threading.Thread(
        target=divide,
        args=(f"Thread-{i}", 10, divisor)
    )
    threads.append(t)
    t.start()

for t in threads:
    t.join()

if errors:
    print("\nErrors collected:")
    for name, err in errors:
        print(f"{name}: {err}")

**Explanation:** A ZeroDivisionError raised inside a thread's target function terminates that thread silently — it never propagates to the main thread's own exception handlers on its own. The only way to surface it is to catch it inside the thread itself and communicate it out through a shared data structure, guarded here by a Lock since multiple threads could otherwise append to the shared errors list at the same instant. Calling .join() on every thread before reading errors guarantees all threads have actually finished before the report is printed.

## Exercise 20. Exception in Generator

**Concept:** gen.throw() to inject an exception into a paused generator

**Problem:** Build a generator that yields file lines, and demonstrate injecting a RuntimeError mid-iteration to cancel it early.

**Given:**
```
a 3-line file; read line 1 normally, then inject a RuntimeError before line 2
```

**Expected Output:**
```
Line 1: Hello from line 1
Generator cancelled: operation aborted
Done.
```

**Hint:** gen.throw(ExcType, msg) resumes the generator at its current yield and raises the exception exactly there.

In [ ]:
def file_lines(filename):
    try:
        f = open(filename, "r")
    except IOError as e:
        print(f"Could not open file: {e}")
        return

    with f:
        for line in f:
            try:
                yield line.rstrip("\n")
            except RuntimeError as e:
                print(f"Generator cancelled: {e}")
                return

with open("sample.txt", "w") as f:
    f.write("Hello from line 1\nHello from line 2\nHello from line 3\n")

gen = file_lines("sample.txt")

line = next(gen)
print(f"Line 1: {line}")

try:
    gen.throw(RuntimeError("operation aborted"))
except StopIteration:
    pass

print("Done.")

**Explanation:** A generator's yield is a genuine suspension point: calling gen.throw(exc) resumes the paused generator and immediately raises that exact exception at the line where it's currently sitting — which is exactly where the try/except RuntimeError around yield catches it. Using return after catching the injected exception makes the generator raise StopIteration to signal it's finished, which is why the calling code wraps gen.throw() in its own try/except StopIteration.